# LeetCode 1807 — Evaluate the Bracket Pairs of a String bằng Python

Link đề: https://leetcode.com/problems/evaluate-the-bracket-pairs-of-a-string/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def evaluate(self, s: str, knowledge: List[List[str]]) -> str:
        ...
```

Lộ trình: **thay thế lặp đi lặp lại trên chuỗi + tra `knowledge` tuyến tính** (baseline) → **bảng băm + quét một lượt với `find`** `O(n + K)` (lời giải nộp) → **máy trạng thái từng ký tự** → **`split('(')`** → **biểu thức chính quy `re.sub`**.

## 1. Mục tiêu học

1. Chuyển danh sách cặp `[key, value]` thành **`dict`** để tra cứu `O(1)` trung bình, và dùng `dict.get(key, "?")` cho giá trị mặc định.
2. Duyệt chuỗi **một lượt** với con trỏ `i`, nhảy qua cả cặp ngoặc bằng `str.find`.
3. Xây chuỗi kết quả bằng **list + `"".join`** thay vì nối `+=` lặp lại hay cắt-dán chuỗi.
4. Biết các cách viết thay thế: **máy trạng thái**, **`split`/`partition`**, **regex với hàm thay thế**.
5. Tránh lỗi lệch chỉ số khi cắt khóa, lỗi `KeyError`, và lỗi "chỉ thay các khóa đã biết".

## 2. Đề bài & ví dụ

Cho chuỗi `s` chứa một số **cặp ngoặc tròn**, mỗi cặp chứa một **khóa** khác rỗng. Ví dụ `"(name)is(age)yearsold"` có hai cặp ngoặc với khóa `"name"` và `"age"`.

Cho mảng `knowledge`, mỗi `knowledge[i] = [key_i, value_i]` cho biết khóa `key_i` có giá trị `value_i`. Khi **đánh giá** một cặp ngoặc chứa khóa `key`:
- thay cả khóa lẫn cặp ngoặc bằng giá trị tương ứng;
- nếu **không biết** giá trị của khóa, thay bằng dấu `"?"`.

Mỗi khóa xuất hiện tối đa một lần trong `knowledge`. Không có ngoặc **lồng nhau**. Trả về chuỗi sau khi đánh giá mọi cặp ngoặc.

**Ví dụ 1:** `s = "(name)is(age)yearsold"`, `knowledge = [["name","bob"],["age","two"]]` → `"bobistwoyearsold"`.

**Ví dụ 2:** `s = "hi(name)"`, `knowledge = [["a","b"]]` → `"hi?"`.

**Ví dụ 3:** `s = "(a)(a)(a)aaa"`, `knowledge = [["a","yes"]]` → `"yesyesyesaaa"` (chữ `a` ngoài ngoặc giữ nguyên).

**Ràng buộc:** `1 ≤ s.length ≤ 10⁵`; `0 ≤ knowledge.length ≤ 10⁵`; `1 ≤ key_i.length, value_i.length ≤ 10`; `s` gồm chữ thường và `(`, `)`; mọi `(` có `)` tương ứng; khóa trong ngoặc khác rỗng; không lồng ngoặc; khóa và giá trị là chữ thường; các `key_i` khác nhau.

In [1]:
import random
import re
import time
from typing import List

examples = [
    ("(name)is(age)yearsold", [["name", "bob"], ["age", "two"]], "bobistwoyearsold"),
    ("hi(name)", [["a", "b"]], "hi?"),
    ("(a)(a)(a)aaa", [["a", "yes"]], "yesyesyesaaa"),
]
for s, kn, want in examples:
    print(repr(s), kn, "->", repr(want))

'(name)is(age)yearsold' [['name', 'bob'], ['age', 'two']] -> 'bobistwoyearsold'
'hi(name)' [['a', 'b']] -> 'hi?'
'(a)(a)(a)aaa' [['a', 'yes']] -> 'yesyesyesaaa'


## 3. Kiến thức nền

### 3.1 Dựng `dict` từ list cặp, và `dict.get` với giá trị mặc định

In [2]:
knowledge = [["name", "bob"], ["age", "two"]]
d = dict(knowledge)              # mỗi phần tử là [key, value]
print(d)
print(d.get("name", "?"), d.get("city", "?"))

{'name': 'bob', 'age': 'two'}
bob ?


### 3.2 `str.find(sub, start)`

Trả về chỉ số xuất hiện đầu tiên của `sub` tính từ `start`, hoặc `-1` nếu không có. Chạy trong `O(khoảng cách tới chỗ tìm thấy)`.

In [3]:
s = "(name)is(age)yearsold"
i = s.find("(")
j = s.find(")", i + 1)
print(i, j, "khóa =", repr(s[i + 1:j]))
print("tìm '(' tiếp theo từ", j + 1, "->", s.find("(", j + 1))

0 5 khóa = 'name'
tìm '(' tiếp theo từ 6 -> 8


### 3.3 Xây chuỗi bằng list + `join`

Chuỗi Python bất biến. Mỗi lần `res = res + x` về lý thuyết tạo chuỗi mới (CPython đôi khi tối ưu, nhưng không bảo đảm). Cách chuẩn: gom các mảnh vào list rồi `"".join(parts)` một lần — `O(tổng độ dài)`.

In [4]:
parts = []
for piece in ["bob", "is", "two", "yearsold"]:
    parts.append(piece)
print("".join(parts))

bobistwoyearsold


### 3.4 `str.split` và `str.partition`

`"a(b)c(d)e".split("(")` → `["a", "b)c", "d)e"]`. `"b)c".partition(")")` → `("b", ")", "c")` — tách đúng một lần ở dấu đầu tiên.

In [5]:
print("a(b)c(d)e".split("("))
print("b)c".partition(")"))

['a', 'b)c', 'd)e']
('b', ')', 'c')


### 3.5 `re.sub` với hàm thay thế

`re.sub(pattern, func, s)` gọi `func(match)` cho mỗi chỗ khớp và thay bằng chuỗi nó trả về. Mẫu `\(([a-z]+)\)` khớp `(` + nhóm chữ thường + `)`; `m.group(1)` là phần trong ngoặc.

In [6]:
d = {"name": "bob"}
print(re.sub(r"\(([a-z]+)\)", lambda m: d.get(m.group(1), "?"), "hi(name)(x)!"))

hibob?!


## 4. Ý tưởng nền tảng / trực giác

Tính tay Ví dụ 1 `s = "(name)is(age)yearsold"`:

```
i=0  '('  -> tìm ')' tại 5, khóa "name" -> "bob";  nhảy i = 6
i=6  'i'  -> giữ
i=7  's'  -> giữ
i=8  '('  -> tìm ')' tại 12, khóa "age"  -> "two";  nhảy i = 13
i=13..20 'yearsold' -> giữ
kết quả: "bob" + "is" + "two" + "yearsold"
```

Hai việc cần làm nhanh:
1. **Tra khóa** → `dict` (thay vì quét `knowledge`).
2. **Ghép kết quả** → list mảnh + `join` (thay vì cắt-dán chuỗi lớn mỗi lần).

Vì không có ngoặc lồng nhau, mỗi `(` khớp với `)` **gần nhất** phía sau.

In [7]:
s = "(name)is(age)yearsold"
for k in range(len(s)):
    if s[k] == "(":
        j = s.find(")", k)
        print(f"cặp ngoặc [{k}, {j}] chứa khóa {s[k+1:j]!r}")

cặp ngoặc [0, 5] chứa khóa 'name'
cặp ngoặc [8, 12] chứa khóa 'age'


## 5. Cách 1 — Brute Force / baseline: cắt-dán chuỗi + tra cứu tuyến tính

**Ý tưởng (làm đúng như mô tả):** lặp: tìm `(` đầu tiên, tìm `)` sau nó, quét `knowledge` để lấy giá trị (hoặc `?`), rồi dựng chuỗi mới `s[:i] + value + s[j+1:]`. Dừng khi hết `(`.

Lưu ý: giá trị chỉ gồm chữ thường nên chuỗi thay vào không tạo ra ngoặc mới; ta tìm `(` tiếp theo **từ sau phần vừa thay** để không đụng lại.

In [8]:
def evaluate_brute(s, knowledge):
    start = 0
    while True:
        i = s.find("(", start)
        if i == -1:
            return s
        j = s.find(")", i)
        key = s[i + 1:j]
        val = "?"
        for k, v in knowledge:              # tra cứu tuyến tính O(K)
            if k == key:
                val = v
                break
        s = s[:i] + val + s[j + 1:]          # dựng lại chuỗi O(n)
        start = i + len(val)

for s, kn, want in examples:
    print(repr(evaluate_brute(s, kn)), "| kỳ vọng", repr(want))

'bobistwoyearsold' | kỳ vọng 'bobistwoyearsold'
'hi?' | kỳ vọng 'hi?'
'yesyesyesaaa' | kỳ vọng 'yesyesyesaaa'


**Độ phức tạp:** gọi `n = len(s)`, `K = len(knowledge)`, `P` = số cặp ngoặc (≤ `n/3`). Mỗi cặp tốn `O(K)` tra cứu + `O(n)` dựng chuỗi ⇒ `O(P · (n + K))`. Worst `n = 10⁵`, `P ≈ 3·10⁴`, `K = 10⁵` ⇒ cỡ `6·10⁹`.

**Hai nút thắt:** tra cứu tuyến tính và dựng lại chuỗi cả chuỗi.

In [9]:
rng = random.Random(0)
keys = ["".join(rng.choice("abcdefghij") for _ in range(6)) for _ in range(5000)]
kn = [[k, "v"] for k in dict.fromkeys(keys)]
s_mid = "".join("(" + rng.choice(keys) + ")x" for _ in range(3000))
t = time.perf_counter(); evaluate_brute(s_mid, kn); dt = time.perf_counter() - t
print(f"n={len(s_mid)}, K={len(kn)}: {dt:.2f}s")

n=27000, K=4992: 0.16s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — `dict`:** khóa là duy nhất ⇒ `d = dict(knowledge)` (`O(K)` một lần), tra cứu `O(len(key))`.

**Insight 2 — một lượt quét:** dùng con trỏ `i` chạy trên `s` gốc (không sửa `s`):
- *Nếu* `s[i] == '('` *thì* `j = s.find(')', i)`; thêm `d.get(s[i+1:j], '?')` vào kết quả; đặt `i = j + 1`.
- *Ngược lại* thêm `s[i]`; `i += 1`.

**Invariant:** sau mỗi bước, `parts` chứa kết quả đánh giá đúng của tiền tố `s[:i]`, và `i` không bao giờ nằm **bên trong** một cặp ngoặc.

Tối ưu thêm: thay vì thêm từng ký tự, thêm cả **đoạn chữ thường** giữa hai cặp ngoặc một lần (`s[i:k]` với `k` là `(` tiếp theo).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `dict` + quét một lượt với `find` (lời giải chuẩn)

In [10]:
def evaluate_scan_demo(s, knowledge):
    d = dict(knowledge)
    parts, i, n = [], 0, len(s)
    while i < n:
        if s[i] == "(":
            j = s.find(")", i)
            key = s[i + 1:j]
            val = d.get(key, "?")
            print(f"i={i:>2}: ngoặc [{i},{j}] khóa {key!r} -> {val!r}")
            parts.append(val)
            i = j + 1
        else:
            k = s.find("(", i)
            if k == -1:
                k = n
            print(f"i={i:>2}: đoạn thường {s[i:k]!r}")
            parts.append(s[i:k])
            i = k
    return "".join(parts)

print("=>", repr(evaluate_scan_demo(*examples[0][:2])))

i= 0: ngoặc [0,5] khóa 'name' -> 'bob'
i= 6: đoạn thường 'is'
i= 8: ngoặc [8,12] khóa 'age' -> 'two'
i=13: đoạn thường 'yearsold'
=> 'bobistwoyearsold'


In [11]:
class SolutionScan:
    def evaluate(self, s: str, knowledge: List[List[str]]) -> str:
        d = dict(knowledge)
        parts, i, n = [], 0, len(s)
        while i < n:
            if s[i] == "(":
                j = s.find(")", i)
                parts.append(d.get(s[i + 1:j], "?"))
                i = j + 1
            else:
                k = s.find("(", i)
                if k == -1:
                    k = n
                parts.append(s[i:k])
                i = k
        return "".join(parts)

for s, kn, want in examples:
    print(SolutionScan().evaluate(s, kn) == want)

True
True
True


**Độ phức tạp:** mỗi ký tự của `s` được `find` đi qua đúng một lần; mỗi khóa cắt và băm một lần ⇒ `O(n + K·L)` với `L ≤ 10` (độ dài khóa/giá trị) — tức `O(n + K)`. Bộ nhớ `O(n + K)`.

### 7.2 Cách 3 — Máy trạng thái từng ký tự

Hai trạng thái: **ngoài ngoặc** (chép ký tự) và **trong ngoặc** (gom khóa vào bộ đệm). Gặp `(` → vào trạng thái trong; gặp `)` → tra khóa, xả bộ đệm, ra ngoài.

In [12]:
class SolutionFSM:
    def evaluate(self, s: str, knowledge: List[List[str]]) -> str:
        d = dict(knowledge)
        out, key, inside = [], [], False
        for c in s:
            if c == "(":
                inside = True
            elif c == ")":
                out.append(d.get("".join(key), "?"))
                key.clear()
                inside = False
            elif inside:
                key.append(c)
            else:
                out.append(c)
        return "".join(out)

for s, kn, want in examples:
    print(SolutionFSM().evaluate(s, kn) == want)

True
True
True


**Độ phức tạp:** `O(n + K)`; dễ mở rộng (ví dụ thêm trạng thái "escape"), nhưng vòng `for` từng ký tự trong Python chậm hơn `find` (chạy bằng C).

### 7.3 Cách 4 — `split('(')` + `partition(')')`

Sau khi tách theo `(`: phần đầu là chữ thường thuần; mỗi phần sau có dạng `key)phần_còn_lại`.

In [13]:
class SolutionSplit:
    def evaluate(self, s: str, knowledge: List[List[str]]) -> str:
        d = dict(knowledge)
        pieces = s.split("(")
        out = [pieces[0]]
        for p in pieces[1:]:
            key, _, rest = p.partition(")")
            out.append(d.get(key, "?"))
            out.append(rest)
        return "".join(out)

for s, kn, want in examples:
    print(SolutionSplit().evaluate(s, kn) == want)

True
True
True


### 7.4 Cách 5 — Biểu thức chính quy

In [14]:
class SolutionRegex:
    PATTERN = re.compile(r"\(([a-z]+)\)")
    def evaluate(self, s: str, knowledge: List[List[str]]) -> str:
        d = dict(knowledge)
        return self.PATTERN.sub(lambda m: d.get(m.group(1), "?"), s)

for s, kn, want in examples:
    print(SolutionRegex().evaluate(s, kn) == want)

True
True
True


Cả hai cách 4, 5 đều `O(n + K)` và rất ngắn; regex đọc rõ ý đồ nhất nhưng cần biết cú pháp. Chú ý thoát ngoặc `\(` `\)` trong mẫu.

## 8. TRACE từng bước

Trace Cách 3 (máy trạng thái) trên Ví dụ 2 `s = "hi(name)"`, `knowledge = [["a","b"]]`:

| ký tự | trạng thái trước | hành động | `key` | `out` |
|---|---|---|---|---|
| h | ngoài | chép | `""` | `h` |
| i | ngoài | chép | `""` | `hi` |
| ( | ngoài | vào ngoặc | `""` | `hi` |
| n,a,m,e | trong | gom khóa | `name` | `hi` |
| ) | trong | tra `name` → không có → `?` | `""` | `hi?` |

In [15]:
def trace_fsm(s, knowledge):
    d = dict(knowledge)
    out, key, inside = [], [], False
    for idx, c in enumerate(s):
        state = "trong" if inside else "ngoài"
        if c == "(":
            inside = True; act = "vào ngoặc"
        elif c == ")":
            k = "".join(key); v = d.get(k, "?")
            out.append(v); key.clear(); inside = False
            act = f"tra {k!r} -> {v!r}"
        elif inside:
            key.append(c); act = "gom khóa"
        else:
            out.append(c); act = "chép"
        print(f"{idx:>2} {c!r} [{state}] {act:<22} key={''.join(key)!r:<8} out={''.join(out)!r}")
    return "".join(out)

trace_fsm("hi(name)", [["a", "b"]])

 0 'h' [ngoài] chép                   key=''       out='h'
 1 'i' [ngoài] chép                   key=''       out='hi'
 2 '(' [ngoài] vào ngoặc              key=''       out='hi'
 3 'n' [trong] gom khóa               key='n'      out='hi'
 4 'a' [trong] gom khóa               key='na'     out='hi'
 5 'm' [trong] gom khóa               key='nam'    out='hi'
 6 'e' [trong] gom khóa               key='name'   out='hi'
 7 ')' [trong] tra 'name' -> '?'      key=''       out='hi?'


'hi?'

In [16]:
trace_fsm("(a)(a)(a)aaa", [["a", "yes"]])

 0 '(' [ngoài] vào ngoặc              key=''       out=''
 1 'a' [trong] gom khóa               key='a'      out=''
 2 ')' [trong] tra 'a' -> 'yes'       key=''       out='yes'
 3 '(' [ngoài] vào ngoặc              key=''       out='yes'
 4 'a' [trong] gom khóa               key='a'      out='yes'
 5 ')' [trong] tra 'a' -> 'yes'       key=''       out='yesyes'
 6 '(' [ngoài] vào ngoặc              key=''       out='yesyes'
 7 'a' [trong] gom khóa               key='a'      out='yesyes'
 8 ')' [trong] tra 'a' -> 'yes'       key=''       out='yesyesyes'
 9 'a' [ngoài] chép                   key=''       out='yesyesyesa'
10 'a' [ngoài] chép                   key=''       out='yesyesyesaa'
11 'a' [ngoài] chép                   key=''       out='yesyesyesaaa'


'yesyesyesaaa'

## 9. Code nộp LeetCode

In [17]:
from typing import List

class Solution:
    def evaluate(self, s: str, knowledge: List[List[str]]) -> str:
        d = dict(knowledge)                 # key -> value, tra O(1) trung bình
        parts = []
        i, n = 0, len(s)
        while i < n:
            if s[i] == "(":
                j = s.find(")", i)          # không lồng ngoặc: ')' gần nhất
                parts.append(d.get(s[i + 1:j], "?"))
                i = j + 1
            else:
                parts.append(s[i])
                i += 1
        return "".join(parts)

for s, kn, want in examples:
    assert Solution().evaluate(s, kn) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Viết `s = T₀ (k₁) T₁ (k₂) T₂ … (k_P) T_P`, trong đó `T_t` là các đoạn chữ thường (có thể rỗng) và `k_t` là các khóa — cách viết này **duy nhất** vì không có ngoặc lồng nhau và mọi `(` đều có `)`. Kết quả đúng theo đề là `T₀ v₁ T₁ v₂ … v_P T_P` với `v_t = d.get(k_t, "?")`.

**Bất biến:** đầu mỗi vòng lặp, `i` là chỉ số đầu của một `T_t` hoặc của một `(`, và `"".join(parts)` bằng kết quả đúng cho `s[:i]`.
- Nếu `s[i]` là chữ: nó thuộc một `T_t`, được chép nguyên — đúng.
- Nếu `s[i] == '('`: vì không lồng ngoặc, `)` đầu tiên sau `i` là `)` của chính cặp này ⇒ `s[i+1:j] = k_t`; thêm `v_t` và nhảy qua `)` — bất biến giữ.

Kết thúc `i = n` ⇒ `parts` là kết quả cho cả chuỗi. Các cách 3–5 cũng dựa trên đúng cách phân rã này.

## 11. Phân tích Time / Space

Gọi `n = len(s)`, `K = len(knowledge)`, `P` = số cặp ngoặc, `L ≤ 10`.

| Cách | Time | Space |
|---|---|---|
| 1. Cắt-dán + tra tuyến tính | `O(P·(n + K·L))` | `O(n)` |
| 2. `dict` + `find` | `O(n + K·L)` | `O(n + K)` |
| 3. Máy trạng thái | `O(n + K·L)` (hằng số lớn hơn) | `O(n + K)` |
| 4. `split` | `O(n + K·L)` | `O(n + K)` |
| 5. Regex | `O(n + K·L)` | `O(n + K)` |

In [18]:
rng = random.Random(1807)
keys = ["".join(rng.choice("abcdefghij") for _ in range(rng.randint(1, 10))) for _ in range(10**5)]
kn_big = [[k, "z" * rng.randint(1, 10)] for k in dict.fromkeys(keys)]
pieces, total_len = [], 0
while total_len < 10**5 - 15:
    piece = "(" + rng.choice(keys) + ")" if rng.random() < 0.5 else rng.choice("xyz")
    pieces.append(piece)
    total_len += len(piece)
s_big = "".join(pieces)
print("n =", len(s_big), "K =", len(kn_big))
outs = []
for name, sol in [("Nộp", Solution()), ("Cách 2", SolutionScan()), ("Cách 3", SolutionFSM()),
                  ("Cách 4", SolutionSplit()), ("Cách 5", SolutionRegex())]:
    t = time.perf_counter()
    outs.append(sol.evaluate(s_big, kn_big))
    print(f"{name}: {1000*(time.perf_counter()-t):.1f} ms")
assert len(set(outs)) == 1

n = 99985 K = 66954


Nộp: 11.8 ms


Cách 2: 9.7 ms
Cách 3: 12.8 ms
Cách 4: 8.9 ms
Cách 5: 9.5 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu baseline)

In [19]:
sols = [Solution(), SolutionScan(), SolutionFSM(), SolutionSplit(), SolutionRegex()]
edge = examples + [
    ("abc", [], "abc"),                          # không ngoặc, knowledge rỗng
    ("(a)", [], "?"),
    ("(a)", [["a", "b"]], "b"),
    ("(ab)(ba)", [["ab", "x"]], "x?"),
    ("x(k)y(k)z", [["k", "kk"]], "xkkykkz"),
    ("(name)", [["nam", "x"], ["name", "y"], ["names", "z"]], "y"),   # khóa gần giống
]
for s, kn, want in edge:
    for sol in sols:
        assert sol.evaluate(s, kn) == want, (s, kn, sol)
    assert evaluate_brute(s, kn) == want
    print("OK", repr(s), "->", repr(want))

OK '(name)is(age)yearsold' -> 'bobistwoyearsold'
OK 'hi(name)' -> 'hi?'
OK '(a)(a)(a)aaa' -> 'yesyesyesaaa'
OK 'abc' -> 'abc'
OK '(a)' -> '?'
OK '(a)' -> 'b'
OK '(ab)(ba)' -> 'x?'
OK 'x(k)y(k)z' -> 'xkkykkz'
OK '(name)' -> 'y'


### 12.2 Ngẫu nhiên có seed: so với baseline

In [20]:
def gen_case(rng):
    alpha = "abc"
    vocab = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 3))) for _ in range(6)]
    known = rng.sample(sorted(set(vocab)), rng.randint(0, len(set(vocab))))
    kn = [[k, "".join(rng.choice("xyz") for _ in range(rng.randint(1, 4)))] for k in known]
    parts = []
    for _ in range(rng.randint(0, 8)):
        if rng.random() < 0.5:
            parts.append("(" + rng.choice(vocab) + ")")
        else:
            parts.append("".join(rng.choice(alpha) for _ in range(rng.randint(1, 3))))
    s = "".join(parts) or "a"
    return s, kn

rng = random.Random(42)
for _ in range(2000):
    s, kn = gen_case(rng)
    want = evaluate_brute(s, kn)
    for sol in sols:
        assert sol.evaluate(s, kn) == want, (s, kn)
print("2000 case ngẫu nhiên: 5 cách khớp baseline")

2000 case ngẫu nhiên: 5 cách khớp baseline


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ `replace` các khóa có trong `knowledge`

In [21]:
def wrong_replace_known(s, knowledge):
    for k, v in knowledge:
        s = s.replace("(" + k + ")", v)        # SAI: khóa không biết vẫn còn nguyên ngoặc
    return s

s, kn = "hi(name)(a)", [["a", "b"]]
print("sai:", repr(wrong_replace_known(s, kn)), "| đúng:", repr(Solution().evaluate(s, kn)))

sai: 'hi(name)b' | đúng: 'hi?b'


**Vì sao sai:** đề yêu cầu thay khóa **không biết** bằng `"?"`; duyệt theo `knowledge` không bao giờ "nhìn thấy" các khóa đó. Ngoài ra mỗi `replace` là `O(n)` ⇒ `O(n·K)`. **Sửa:** duyệt theo **các cặp ngoặc trong `s`**, tra `dict`.

### Lỗi 2 — Cắt khóa lệch một ký tự

In [22]:
def wrong_slice(s, knowledge):
    d = dict(knowledge); parts = []; i = 0
    while i < len(s):
        if s[i] == "(":
            j = s.find(")", i)
            parts.append(d.get(s[i:j], "?"))     # SAI: lấy cả '('
            i = j + 1
        else:
            parts.append(s[i]); i += 1
    return "".join(parts)

s, kn = examples[0][:2]
print("sai:", repr(wrong_slice(s, kn)), "| đúng:", repr(Solution().evaluate(s, kn)))

sai: '?is?yearsold' | đúng: 'bobistwoyearsold'


**Vì sao sai:** `s[i:j]` là `"(name"`, không có trong `dict` ⇒ mọi cặp thành `"?"`. **Sửa:** `s[i+1:j]` (bỏ `(` ở `i`, lát cắt đã không gồm `)` ở `j`).

### Lỗi 3 — Dùng `d[key]` thay vì `d.get(key, "?")`

In [23]:
def wrong_keyerror(s, knowledge):
    d = dict(knowledge); parts = []; i = 0
    while i < len(s):
        if s[i] == "(":
            j = s.find(")", i)
            parts.append(d[s[i + 1:j]])          # SAI: KeyError với khóa không biết
            i = j + 1
        else:
            parts.append(s[i]); i += 1
    return "".join(parts)

try:
    print(wrong_keyerror("hi(name)", [["a", "b"]]))
except KeyError as e:
    print("KeyError:", e, "| đúng:", repr(Solution().evaluate("hi(name)", [["a", "b"]])))

KeyError: 'name' | đúng: 'hi?'


**Vì sao sai:** khóa không có trong `dict` ⇒ ngoại lệ. **Sửa:** `d.get(key, "?")`.

### Lỗi 4 — Tra `knowledge` tuyến tính cho mỗi cặp ngoặc

In [24]:
def slow_linear_lookup(s, knowledge):
    parts = []; i = 0
    while i < len(s):
        if s[i] == "(":
            j = s.find(")", i); key = s[i + 1:j]
            parts.append(next((v for k, v in knowledge if k == key), "?"))   # CHẬM: O(K)
            i = j + 1
        else:
            parts.append(s[i]); i += 1
    return "".join(parts)

cut = s_big.rfind(")", 0, 3000) + 1            # cắt ở ranh giới cặp ngoặc
small_s, small_kn = s_big[:cut], kn_big[:20000]
t = time.perf_counter(); r1 = slow_linear_lookup(small_s, small_kn); dt = time.perf_counter() - t
assert r1 == Solution().evaluate(small_s, small_kn)
factor = (len(s_big) / len(small_s)) * (len(kn_big) / len(small_kn))
print(f"n={len(small_s)}, K=2·10^4: {dt:.2f}s (đúng nhưng chậm) — n=K=10^5 ước tính ≈ {dt * factor:.0f}s")

n=2991, K=2·10^4: 0.09s (đúng nhưng chậm) — n=K=10^5 ước tính ≈ 10s

**Vì sao sai:** đúng kết quả nhưng `O(P·K)` — với `P ≈ 3·10⁴`, `K = 10⁵` là TLE. **Sửa:** `dict(knowledge)` một lần.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Thay thế placeholder trong template"** (`(key)`, `{key}`, `${key}`) ⇒ `dict` + quét một lượt (hoặc regex).
- Tra cứu nhiều lần theo khóa ⇒ **luôn** dựng `dict` trước.
- Kết quả là chuỗi ghép nhiều mảnh ⇒ list + `"".join`.
- "Không lồng ngoặc" ⇒ `)` gần nhất là đúng cặp; có lồng ⇒ cần **stack** (Bài tập 5).
- `dict.get(k, default)` để tránh `KeyError`.

## 15. Biến thể của bài

1. **Ngoặc lồng nhau**, khóa bên ngoài được tạo từ kết quả bên trong ⇒ stack (Bài tập 5); tương tự LeetCode 1190 *Reverse Substrings Between Each Pair of Parentheses*.
2. **Giá trị có thể chứa ngoặc** và phải đánh giá đệ quy ⇒ cần phát hiện chu trình (khóa tự tham chiếu).
3. **Template kiểu `{{key}}`** với dấu mở/đóng nhiều ký tự ⇒ `find` với chuỗi con nhiều ký tự hoặc regex.
4. **Đếm số khóa không biết** hoặc trả về danh sách khóa thiếu.
5. LeetCode 2194/2810… các bài xử lý chuỗi tuyến tính; LeetCode 1678 *Goal Parser Interpretation* (bản dễ).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cắt-dán + tra tuyến tính | làm đúng câu chữ | `O(P(n+K))` | `O(n)` | kiểm chứng |
| 2. `dict` + `find` | nhảy qua cặp ngoặc | `O(n+K)` | `O(n+K)` | **nộp bài** |
| 3. Máy trạng thái | ngoài/trong ngoặc | `O(n+K)` | `O(n+K)` | dễ mở rộng luật |
| 4. `split`/`partition` | tách theo `(` | `O(n+K)` | `O(n+K)` | viết nhanh |
| 5. Regex | `re.sub` + hàm | `O(n+K)` | `O(n+K)` | gọn nhất, khi quen regex |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `evaluate("(x)(y)z", [["y","b"]])` và `evaluate("((a)", ...)` — câu thứ hai có hợp lệ theo ràng buộc không?

**Bài 2 (tính tay):** Trace Cách 2 trên `s = "a(b)c(d)"`, `knowledge = [["d","q"]]`: ghi `i`, mảnh được thêm.

**Bài 3 (điền TODO):** Viết `extract_keys(s)` trả về list các khóa theo thứ tự xuất hiện.

**Bài 4 (tự viết lại):** Viết `missing_keys(s, knowledge)` trả về **tập** các khóa xuất hiện trong `s` mà không có trong `knowledge`.

**Bài 5 (mini challenge — ngoặc lồng):** Bỏ ràng buộc "không lồng ngoặc". Đánh giá từ trong ra ngoài: kết quả của cặp trong trở thành một phần **khóa** của cặp ngoài. Ví dụ `"((a)b)"` với `a → x`, `xb → ok` cho `"ok"`. Viết `evaluate_nested(s, knowledge)` bằng **stack**, `O(n · L)`.

In [25]:
def extract_keys(s):
    # TODO
    pass

def missing_keys(s, knowledge):
    # TODO
    pass

def evaluate_nested(s, knowledge):
    # TODO: stack các list ký tự
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `x` không biết | Kết quả `"?bz"` | `"((a)"` vi phạm ràng buộc (lồng + thiếu `)`) |
| 2 | `i=0` chép `a` | `i=1` gặp `(`, khóa `b` → `?` | Kết quả `"a?cq"` |
| 3 | `re.findall(r"\(([a-z]+)\)", s)` | Hoặc dùng `find` như Cách 2 | Giữ thứ tự xuất hiện |
| 4 | `set(extract_keys(s))` | Trừ `set` khóa trong `knowledge` | `{k for k in keys if k not in d}` |
| 5 | Stack bắt đầu `[[]]` | Gặp `(` push list mới; gặp `)` pop, tra khóa, thêm giá trị vào list đỉnh | Kết quả là `"".join(stack[0])` |

## 19. Đáp án tham khảo

In [26]:
print("Bài 1:", repr(Solution().evaluate("(x)(y)z", [["y", "b"]])))
assert Solution().evaluate("(x)(y)z", [["y", "b"]]) == "?bz"
print("Bài 2:"); r2 = evaluate_scan_demo("a(b)c(d)", [["d", "q"]]); assert r2 == "a?cq"; print("=>", r2)

Bài 1: '?bz'
Bài 2:
i= 0: đoạn thường 'a'
i= 1: ngoặc [1,3] khóa 'b' -> '?'
i= 4: đoạn thường 'c'
i= 5: ngoặc [5,7] khóa 'd' -> 'q'
=> a?cq


In [27]:
def extract_keys(s):
    keys, i = [], 0
    while True:
        i = s.find("(", i)
        if i == -1:
            return keys
        j = s.find(")", i)
        keys.append(s[i + 1:j])
        i = j + 1

def missing_keys(s, knowledge):
    d = dict(knowledge)
    return {k for k in extract_keys(s) if k not in d}

rng = random.Random(4)
for _ in range(500):
    s, kn = gen_case(rng)
    assert extract_keys(s) == re.findall(r"\(([a-z]+)\)", s)
    assert missing_keys(s, kn) == set(re.findall(r"\(([a-z]+)\)", s)) - {k for k, _ in kn}
print("Bài 3, 4: OK", extract_keys("(name)is(age)yearsold"), missing_keys("hi(name)(a)", [["a", "b"]]))

Bài 3, 4: OK ['name', 'age'] {'name'}


In [28]:
def evaluate_nested(s, knowledge):
    d = dict(knowledge)
    stack = [[]]
    for c in s:
        if c == "(":
            stack.append([])
        elif c == ")":
            key = "".join(stack.pop())
            stack[-1].append(d.get(key, "?"))
        else:
            stack[-1].append(c)
    return "".join(stack[0])

def evaluate_nested_brute(s, knowledge):
    d = dict(knowledge)
    pat = re.compile(r"\(([^()]*)\)")          # cặp ngoặc trong cùng
    while "(" in s:
        s = pat.sub(lambda m: d.get(m.group(1), "?"), s, count=1)
    return s

assert evaluate_nested("((a)b)", [["a", "x"], ["xb", "ok"]]) == "ok"
assert evaluate_nested("((a)b)", [["a", "x"]]) == "?"
rng = random.Random(21)
def gen_nested(rng, depth=0):
    out = []
    for _ in range(rng.randint(1, 3)):
        r = rng.random()
        if r < 0.3 and depth < 3:
            out.append("(" + gen_nested(rng, depth + 1) + ")")
        else:
            out.append(rng.choice(["a", "b", "x", "ab"]))
    return "".join(out)
for _ in range(1000):
    s = gen_nested(rng)
    kn = [[k, rng.choice(["a", "b", "x", "ab"])] for k in ["a", "b", "x", "ab", "xb", "ba", "bx"] if rng.random() < 0.5]
    assert evaluate_nested(s, kn) == evaluate_nested_brute(s, kn), (s, kn)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
d = dict(knowledge)
parts, i = [], 0
while i < len(s):
    if s[i] == "(":
        j = s.find(")", i)
        parts.append(d.get(s[i+1:j], "?"))
        i = j + 1
    else:
        parts.append(s[i]); i += 1
return "".join(parts)
```

**Checklist:**
- [ ] Dựng `dict` **một lần**, không tra `knowledge` tuyến tính?
- [ ] Khóa là `s[i+1:j]` (bỏ cả hai ngoặc)?
- [ ] Khóa không biết → `"?"` bằng `d.get(..., "?")`?
- [ ] Ghép bằng `"".join`, không cắt-dán chuỗi trong vòng lặp?
- [ ] Chữ ngoài ngoặc (kể cả trùng tên khóa) được giữ nguyên?